### Silver data layer transformation


### loading data

In [0]:
dbutils.secrets.listScopes()

[]

In [0]:
display(dbutils.fs.ls("abfss://bronze@azuredatalake90.dfs.core.windows.net/"))

Tried to attach usage logger `pyspark.databricks.pandas.usage_logger`, but an exception was raised: JVM wasn't initialised. Did you call it on executor side?


path,name,size,modificationTime
abfss://bronze@azuredatalake90.dfs.core.windows.net/category/,category/,0,1794997397000
abfss://bronze@azuredatalake90.dfs.core.windows.net/products/,products/,0,1795016988000
abfss://bronze@azuredatalake90.dfs.core.windows.net/sales/,sales/,0,1795016990000
abfss://bronze@azuredatalake90.dfs.core.windows.net/stores/,stores/,0,1795016988000
abfss://bronze@azuredatalake90.dfs.core.windows.net/warranty/,warranty/,0,1795016988000


In [0]:
display(dbutils.fs.ls("abfss://bronze@azuredatalake90.dfs.core.windows.net/"))

path,name,size,modificationTime
abfss://bronze@azuredatalake90.dfs.core.windows.net/category/,category/,0,1794997397000
abfss://bronze@azuredatalake90.dfs.core.windows.net/products/,products/,0,1795016988000
abfss://bronze@azuredatalake90.dfs.core.windows.net/sales/,sales/,0,1795016990000
abfss://bronze@azuredatalake90.dfs.core.windows.net/stores/,stores/,0,1795016988000
abfss://bronze@azuredatalake90.dfs.core.windows.net/warranty/,warranty/,0,1795016988000


In [0]:
display(dbutils.fs.ls(
    "abfss://bronze@azuredatalake90.dfs.core.windows.net/sales/"
))

path,name,size,modificationTime
abfss://bronze@azuredatalake90.dfs.core.windows.net/sales/sales.csv,sales.csv,35250073,1795016991000


In [0]:
sales_path = "abfss://bronze@azuredatalake90.dfs.core.windows.net/sales/sales.csv"

df = spark.read.option("header", True).option("inferSchema", True).csv(sales_path)

display(df.limit(10))

sale_id,sale_date,store_id,product_id,quantity
YG-8782,2023-06-16,ST-10,P-38,10
QX-999001,2022-04-13,ST-63,P-48,10
JG-46890,2021-07-05,ST-26,P-79,5
XJ-1731,2022-07-20,ST-15,P-24,9
FG-95080,2022-03-18,ST-35,P-69,7
II-3954,2024-04-19,ST-28,P-45,10
ES-7238,2020-05-11,ST-3,P-24,6
YO-701453,2023-03-27,ST-73,P-9,7
JB-46303,2024-08-31,ST-14,P-29,5
QY-9761,2021-07-14,ST-44,P-24,1


In [0]:
# Dataset dimensions
print("Rows:", df.count())
print("Columns:", len(df.columns))

# Column names and data types
df.printSchema()

# Missing values per column
from pyspark.sql.functions import col, sum, when

missing_values = df.select([
    sum(when(col(c).isNull(), 1).otherwise(0)).alias(c)
    for c in df.columns
])

display(missing_values)

Rows: 1040200
Columns: 5
root
 |-- sale_id: string (nullable = true)
 |-- sale_date: date (nullable = true)
 |-- store_id: string (nullable = true)
 |-- product_id: string (nullable = true)
 |-- quantity: integer (nullable = true)



sale_id,sale_date,store_id,product_id,quantity
0,0,0,0,0


In [0]:
# Check duplicate sale IDs
total_rows = df.count()
unique_sales = df.select("sale_id").distinct().count()

print("Total Rows:", total_rows)
print("Unique Sale IDs:", unique_sales)
print("Duplicate Sale IDs:", total_rows - unique_sales)

Total Rows: 1040200
Unique Sale IDs: 1040200
Duplicate Sale IDs: 0


In [0]:
from pyspark.sql.functions import col, min, max, avg

df.select(
    min("quantity").alias("Minimum Quantity"),
    max("quantity").alias("Maximum Quantity"),
    avg("quantity").alias("Average Quantity")
).show()

print("Zero Quantity:", df.filter(col("quantity") == 0).count())
print("Negative Quantity:", df.filter(col("quantity") < 0).count())

+----------------+----------------+-----------------+
|Minimum Quantity|Maximum Quantity| Average Quantity|
+----------------+----------------+-----------------+
|               1|              10|5.500234570274947|
+----------------+----------------+-----------------+

Zero Quantity: 0
Negative Quantity: 0


In [0]:
from pyspark.sql.functions import min, max, countDistinct

df.select(
    min("sale_date").alias("Earliest Date"),
    max("sale_date").alias("Latest Date"),
    countDistinct("sale_date").alias("Unique Dates")
).show()


+-------------+-----------+------------+
|Earliest Date|Latest Date|Unique Dates|
+-------------+-----------+------------+
|   2020-01-01| 2024-11-12|        1778|
+-------------+-----------+------------+



In [0]:
silver_path = "abfss://silver@azuredatalake90.dfs.core.windows.net/sales/"

df.write \
    .format("parquet") \
    .mode("overwrite") \
    .save(silver_path)

print("Sales data successfully saved to Silver!")

Sales data successfully saved to Silver!


In [0]:
silver_df = spark.read.parquet(silver_path)

print("Silver Rows:", silver_df.count())
print("Silver Columns:", len(silver_df.columns))

display(silver_df.limit(10))

Silver Rows: 1040200
Silver Columns: 5


sale_id,sale_date,store_id,product_id,quantity
SM-3866,2020-04-15,ST-2,P-1,6
ZP-48493,2021-11-08,ST-23,P-59,4
XB-61634,2024-01-08,ST-17,P-81,8
KT-23841,2024-06-03,ST-52,P-16,7
RP-9902,2022-08-18,ST-13,P-85,2
NZ-82921,2020-08-07,ST-7,P-42,10
UZ-9610,2024-11-10,ST-46,P-24,8
PT-5328,2024-07-29,ST-55,P-12,5
OS-58064,2020-11-09,ST-40,P-33,7
XE-64980,2022-10-26,ST-63,P-33,6


In [0]:
base_path = "abfss://bronze@azuredatalake90.dfs.core.windows.net/"

for folder in ["category", "products", "stores", "warranty"]:
    print(f"\n--- {folder.upper()} ---")
    display(dbutils.fs.ls(base_path + folder + "/"))


--- CATEGORY ---


path,name,size,modificationTime
abfss://bronze@azuredatalake90.dfs.core.windows.net/category/category.csv,category.csv,210,1795016988000
abfss://bronze@azuredatalake90.dfs.core.windows.net/category/parameters/,parameters/,0,1795003509000



--- PRODUCTS ---


path,name,size,modificationTime
abfss://bronze@azuredatalake90.dfs.core.windows.net/products/products.csv,products.csv,4082,1795016988000



--- STORES ---


path,name,size,modificationTime
abfss://bronze@azuredatalake90.dfs.core.windows.net/stores/stores.csv,stores.csv,3266,1795016988000



--- WARRANTY ---


path,name,size,modificationTime
abfss://bronze@azuredatalake90.dfs.core.windows.net/warranty/warranty.csv,warranty.csv,1182597,1795016988000


In [0]:
category_path = "abfss://bronze@azuredatalake90.dfs.core.windows.net/category/category.csv"

category_df = spark.read.option("header", True).option("inferSchema", True).csv(category_path)

display(category_df)

category_id,category_name
CAT-1,Laptop
CAT-2,Audio
CAT-3,Tablet
CAT-4,Smartphone
CAT-5,Wearable
CAT-6,Streaming Device
CAT-7,Desktop
CAT-8,Subscription Service
CAT-9,Smart Speaker
CAT-10,Accessories


In [0]:
products_path = "abfss://bronze@azuredatalake90.dfs.core.windows.net/products/products.csv"

products_df = spark.read.option("header", True).option("inferSchema", True).csv(products_path)

display(products_df.limit(10))

Product_ID,Product_Name,Category_ID,Launch_Date,Price
P-1,MacBook,CAT-1,2023-09-17,1149
P-2,MacBook Air (M1),CAT-1,2023-11-11,1783
P-3,MacBook Air (M2),CAT-1,2020-05-24,1588
P-4,MacBook Pro 13-inch,CAT-1,2021-01-17,1351
P-5,MacBook Pro 14-inch,CAT-1,2024-05-12,768
P-6,MacBook Pro 16-inch,CAT-1,2021-12-28,1179
P-7,MacBook (Retina),CAT-1,2023-10-17,231
P-8,MacBook Air (Retina),CAT-1,2020-01-16,1847
P-9,MacBook Pro (Touch Bar),CAT-1,2024-06-17,1304
P-10,MacBook (Early 2015),CAT-1,2022-11-19,741


In [0]:
from pyspark.sql.functions import col, trim

silver_category = (
    category_df
    .dropDuplicates(['category_id'])
    .filter(col('category_id').isNotNull())
    .withColumn('category_name', trim(col('category_name')))
)

display(silver_category)

category_id,category_name
CAT-3,Tablet
CAT-1,Laptop
CAT-10,Accessories
CAT-2,Audio
CAT-5,Wearable
CAT-6,Streaming Device
CAT-8,Subscription Service
CAT-4,Smartphone
CAT-7,Desktop
CAT-9,Smart Speaker


In [0]:
silver_category.write \
    .format("parquet") \
    .mode("overwrite") \
    .save("abfss://silver@azuredatalake90.dfs.core.windows.net/category/")

In [0]:
silver_products = (
    products_df
    .dropDuplicates(['Product_ID'])
    .filter(col('Product_ID').isNotNull())
    .withColumn('Product_Name', trim(col('Product_Name')))
    .withColumn('Category_ID', trim(col('Category_ID')))
)

In [0]:
silver_products.write \
    .format("parquet") \
    .mode("overwrite") \
    .save("abfss://silver@azuredatalake90.dfs.core.windows.net/products/")

In [0]:
sales_path = "abfss://bronze@azuredatalake90.dfs.core.windows.net/sales/sales.csv"

sales_df = spark.read.option("header", True).option("inferSchema", True).csv(sales_path)

print(sales_df.count())

1040200


In [0]:
silver_sales = (
    sales_df
    .dropDuplicates(['sale_id'])
    .filter(col('sale_id').isNotNull())
    .filter(col('quantity').isNotNull())
    .filter(col('quantity') > 0)
)

display(silver_sales.limit(10))

sale_id,sale_date,store_id,product_id,quantity
AY-284049,2021-08-04,ST-24,P-76,6
HF-13620,2022-11-01,ST-19,P-22,10
CJ-922471,2020-04-24,ST-73,P-22,4
IP-49195,2020-06-30,ST-37,P-16,6
FH-80703,2023-11-25,ST-73,P-71,4
LH-9179,2022-10-03,ST-73,P-57,6
CN-090119,2024-04-21,ST-5,P-38,2
YW-448639,2021-03-10,ST-25,P-89,1
KH-48512,2020-11-11,ST-11,P-36,4
XB-78686,2024-03-24,ST-43,P-10,7


In [0]:
silver_sales.write \
    .format("parquet") \
    .mode("overwrite") \
    .save("abfss://silver@azuredatalake90.dfs.core.windows.net/sales/")

In [0]:
stores_path = "abfss://bronze@azuredatalake90.dfs.core.windows.net/stores/stores.csv"

stores_df = spark.read.option("header", True).option("inferSchema", True).csv(stores_path)

silver_stores = (
    stores_df
    .dropDuplicates(['Store_ID'])
    .filter(col('Store_ID').isNotNull())
    .withColumn('Store_Name', trim(col('Store_Name')))
    .withColumn('City', trim(col('City')))
    .withColumn('Country', trim(col('Country')))
)

display(silver_stores)

Store_ID,Store_Name,City,Country
ST-12,Apple North Michigan Avenue,Chicago,United States
ST-25,Apple Schildergasse,Cologne,Germany
ST-35,Apple Shanghai IFC,Shanghai,China
ST-45,Apple Jewel Changi Airport,Singapore,Singapore
ST-60,Apple Antara,Mexico City,Mexico
ST-15,Apple Downtown Brooklyn,Brooklyn,United States
ST-43,Apple Cotai Central,Macau,China
ST-46,Apple Eaton Centre,Toronto,Canada
ST-53,Apple Brisbane,Brisbane,Australia
ST-68,Apple Champs-Elysees,Paris,France


In [0]:
silver_stores.write \
    .format("parquet") \
    .mode("overwrite") \
    .save("abfss://silver@azuredatalake90.dfs.core.windows.net/stores/")

In [0]:
warranty_path = "abfss://bronze@azuredatalake90.dfs.core.windows.net/warranty/warranty.csv"

warranty_df = spark.read.option("header", True).option("inferSchema", True).csv(warranty_path)

silver_warranty = (
    warranty_df
    .dropDuplicates(['claim_id'])
    .filter(col('claim_id').isNotNull())
    .filter(col('sale_id').isNotNull())
    .withColumn('repair_status', trim(col('repair_status')))
)

display(silver_warranty.limit(10))

claim_id,claim_date,sale_id,repair_status
CL-7124,2024-01-04,UN-941274,Completed
CL-6383,2024-02-05,LH-1755,Pending
CL-9801,2024-02-17,KE-5276,Pending
CL-6811,2024-08-30,SO-324887,Pending
CL-3218,2024-01-16,VV-767827,In Progress
CL-41107,2024-02-05,AQ-5248,Pending
CL-88527,2024-08-14,TH-49294,In Progress
CL-28295,2024-09-21,KG-09039,In Progress
CL-68769,2024-07-11,PX-085809,In Progress
CL-63454,2024-09-13,GX-762341,In Progress


In [0]:
silver_warranty.write \
    .format("parquet") \
    .mode("overwrite") \
    .save("abfss://silver@azuredatalake90.dfs.core.windows.net/warranty/")

In [0]:
silver_path = "abfss://silver@azuredatalake90.dfs.core.windows.net/"

display(dbutils.fs.ls(silver_path))

path,name,size,modificationTime
abfss://silver@azuredatalake90.dfs.core.windows.net/category/,category/,0,1795028919000
abfss://silver@azuredatalake90.dfs.core.windows.net/products/,products/,0,1795028955000
abfss://silver@azuredatalake90.dfs.core.windows.net/sales/,sales/,0,1795028414000
abfss://silver@azuredatalake90.dfs.core.windows.net/stores/,stores/,0,1795029077000
abfss://silver@azuredatalake90.dfs.core.windows.net/warranty/,warranty/,0,1795029145000


### Gold Layer – Product Sales Aggregation

In [0]:
from pyspark.sql.functions import sum, count, avg

gold_product_sales = (
    silver_sales
    .groupBy("product_id")
    .agg(
        sum("quantity").alias("total_quantity_sold"),
        count("sale_id").alias("total_sales"),
        avg("quantity").alias("avg_quantity_per_sale")
    )
)

display(gold_product_sales.limit(10))

product_id,total_quantity_sold,total_sales,avg_quantity_per_sale
P-72,64143,11714,5.4757555062318595
P-46,65085,11778,5.525980641874682
P-15,64844,11742,5.522398228581162
P-66,64594,11653,5.5431219428473355
P-86,64149,11679,5.492679167736964
P-9,63691,11566,5.506743904547813
P-17,64410,11706,5.502306509482317
P-30,64827,11784,5.501272912423625
P-80,63467,11584,5.478850138121547
P-10,63985,11629,5.502192793877375


In [0]:
gold_product_sales.write \
    .format("parquet") \
    .mode("overwrite") \
    .save("abfss://gold@azuredatalake90.dfs.core.windows.net/product_sales/")

In [0]:
gold_df = spark.read.parquet(
    "abfss://gold@azuredatalake90.dfs.core.windows.net/product_sales/"
)

display(gold_df.limit(10))

product_id,total_quantity_sold,total_sales,avg_quantity_per_sale
P-72,64143,11714,5.4757555062318595
P-46,65085,11778,5.525980641874682
P-15,64844,11742,5.522398228581162
P-66,64594,11653,5.5431219428473355
P-86,64149,11679,5.492679167736964
P-9,63691,11566,5.506743904547813
P-17,64410,11706,5.502306509482317
P-30,64827,11784,5.501272912423625
P-80,63467,11584,5.478850138121547
P-10,63985,11629,5.502192793877375


In [0]:
gold_path = "abfss://gold@azuredatalake90.dfs.core.windows.net/"

display(dbutils.fs.ls(gold_path))

path,name,size,modificationTime
abfss://gold@azuredatalake90.dfs.core.windows.net/product_sales/,product_sales/,0,1795029492000


In [0]:
gold_product_sales.write \
    .format("parquet") \
    .mode("overwrite") \
    .save(gold_path + "product_sales/")

In [0]:
gold_df = spark.read.parquet(gold_path + "product_sales/")

display(gold_df)

product_id,total_quantity_sold,total_sales,avg_quantity_per_sale
P-72,64143,11714,5.4757555062318595
P-46,65085,11778,5.525980641874682
P-15,64844,11742,5.522398228581162
P-66,64594,11653,5.5431219428473355
P-86,64149,11679,5.492679167736964
P-9,63691,11566,5.506743904547813
P-17,64410,11706,5.502306509482317
P-30,64827,11784,5.501272912423625
P-80,63467,11584,5.478850138121547
P-10,63985,11629,5.502192793877375


In [0]:
from pyspark.sql.functions import sum, count

gold_category_sales = (
    silver_sales
    .join(
        silver_products,
        silver_sales.product_id == silver_products.Product_ID,
        "inner"
    )
    .join(
        silver_category,
        silver_products.Category_ID == silver_category.category_id,
        "inner"
    )
    .groupBy("category_id", "category_name")
    .agg(
        sum("quantity").alias("total_quantity_sold"),
        count("sale_id").alias("total_sales")
    )
)

display(gold_category_sales)

category_id,category_name,total_quantity_sold,total_sales
CAT-4,Smartphone,834006,151636
CAT-3,Tablet,645303,117142
CAT-10,Accessories,899734,163849
CAT-1,Laptop,640336,116369
CAT-6,Streaming Device,192534,35014
CAT-8,Subscription Service,447062,81561
CAT-9,Smart Speaker,130430,23565
CAT-5,Wearable,582445,105521
CAT-2,Audio,707354,128731
CAT-7,Desktop,642140,116812


In [0]:
gold_category_sales.write \
    .format("parquet") \
    .mode("overwrite") \
    .save(gold_path + "category_sales/")

In [0]:
display(
    spark.read.parquet(gold_path + "category_sales/")
)

category_id,category_name,total_quantity_sold,total_sales
CAT-4,Smartphone,834006,151636
CAT-3,Tablet,645303,117142
CAT-10,Accessories,899734,163849
CAT-1,Laptop,640336,116369
CAT-6,Streaming Device,192534,35014
CAT-8,Subscription Service,447062,81561
CAT-9,Smart Speaker,130430,23565
CAT-5,Wearable,582445,105521
CAT-2,Audio,707354,128731
CAT-7,Desktop,642140,116812


In [0]:
from pyspark.sql.functions import sum, count, col, trim

s = silver_sales.alias("s")
st = silver_stores.alias("st")

gold_store_sales = (
    s.join(
        st,
        col("s.store_id") == col("st.Store_ID"),
        "inner"
    )
    .groupBy(
        col("st.Store_ID"),
        col("st.Store_Name"),
        col("st.City"),
        col("st.Country")
    )
    .agg(
        sum(col("s.quantity")).alias("total_quantity_sold"),
        count(col("s.sale_id")).alias("total_sales")
    )
)

display(gold_store_sales)

Store_ID,Store_Name,City,Country,total_quantity_sold,total_sales
ST-59,Apple Mall of the Emirates,Dubai,UAE,75767,13764
ST-38,Apple Taikoo Li,Chengdu,China,75950,13797
ST-52,Apple Chadstone,Melbourne,Australia,77170,14023
ST-17,Apple Covent Garden,London,United Kingdom,75630,13791
ST-10,Apple South Coast Plaza,Costa Mesa,United States,75815,13718
ST-45,Apple Jewel Changi Airport,Singapore,Singapore,76712,14004
ST-20,Apple Kaerntner Strasse,Vienna,Austria,75965,13771
ST-34,Apple Fukuoka,Fukuoka,Japan,77787,14068
ST-5,Apple SoHo,New York,United States,77186,14059
ST-21,Apple Passeig de Gracia,Barcelona,Spain,74977,13686


In [0]:
gold_store_sales.write \
    .format("parquet") \
    .mode("overwrite") \
    .save(gold_path + "store_sales/")

In [0]:
gold_store_df = spark.read.parquet(gold_path + "store_sales/")

display(gold_store_df)

Store_ID,Store_Name,City,Country,total_quantity_sold,total_sales
ST-59,Apple Mall of the Emirates,Dubai,UAE,75767,13764
ST-38,Apple Taikoo Li,Chengdu,China,75950,13797
ST-52,Apple Chadstone,Melbourne,Australia,77170,14023
ST-17,Apple Covent Garden,London,United Kingdom,75630,13791
ST-10,Apple South Coast Plaza,Costa Mesa,United States,75815,13718
ST-45,Apple Jewel Changi Airport,Singapore,Singapore,76712,14004
ST-20,Apple Kaerntner Strasse,Vienna,Austria,75965,13771
ST-34,Apple Fukuoka,Fukuoka,Japan,77787,14068
ST-5,Apple SoHo,New York,United States,77186,14059
ST-21,Apple Passeig de Gracia,Barcelona,Spain,74977,13686


In [0]:
from pyspark.sql.functions import col, count

gold_warranty_analysis = (
    silver_warranty
    .groupBy("repair_status")
    .agg(
        count("claim_id").alias("total_claims")
    )
)

display(gold_warranty_analysis)

repair_status,total_claims
Completed,7466
Pending,7566
In Progress,7611
Rejected,7357


In [0]:
gold_warranty_analysis.write \
    .format("parquet") \
    .mode("overwrite") \
    .save(gold_path + "warranty_analysis/")

In [0]:
display(
    spark.read.parquet(gold_path + "warranty_analysis/")
)

repair_status,total_claims
Pending,7566
Rejected,7357
In Progress,7611
Completed,7466


In [0]:
## final validation

gold_tables = [
    "product_sales",
    "category_sales",
    "store_sales",
    "warranty_analysis"
]

for table in gold_tables:
    path = gold_path + table + "/"
    df = spark.read.parquet(path)

    print(f"{table}: {df.count()} rows")

product_sales: 89 rows
category_sales: 10 rows
store_sales: 75 rows
warranty_analysis: 4 rows


### quality checks


In [0]:
from pyspark.sql.functions import col

s = silver_sales.select("product_id").distinct().alias("s")
p = silver_products.select("Product_ID").distinct().alias("p")

sales_products_check = (
    s.join(
        p,
        col("s.product_id") == col("p.Product_ID"),
        "left_anti"
    )
)

print("Products missing from Products table:", sales_products_check.count())

display(sales_products_check)

Products missing from Products table: 0


product_id


In [0]:
s = silver_sales.select("store_id").distinct().alias("s")
st = silver_stores.select("Store_ID").distinct().alias("st")

sales_stores_check = (
    s.join(
        st,
        col("s.store_id") == col("st.Store_ID"),
        "left_anti"
    )
)

print("Stores missing from Stores table:", sales_stores_check.count())

display(sales_stores_check)

Stores missing from Stores table: 0


store_id


In [0]:
p = silver_products.select("Category_ID").distinct().alias("p")
c = silver_category.select("category_id").distinct().alias("c")

products_category_check = (
    p.join(
        c,
        col("p.Category_ID") == col("c.category_id"),
        "left_anti"
    )
)

print("Categories missing from Category table:", products_category_check.count())

display(products_category_check)

Categories missing from Category table: 0


Category_ID


In [0]:
w = silver_warranty.select("sale_id").distinct().alias("w")
s = silver_sales.select("sale_id").distinct().alias("s")

warranty_sales_check = (
    w.join(
        s,
        col("w.sale_id") == col("s.sale_id"),
        "left_anti"
    )
)

print("Warranty claims with missing sales:", warranty_sales_check.count())

display(warranty_sales_check)

Warranty claims with missing sales: 0


sale_id


### final project summary


In [0]:
print("========== AZURE DATA ENGINEERING PROJECT ==========")

print("\nBRONZE LAYER")
print("5 original CSV files")

print("\nSILVER LAYER")
for name, df in [
    ("Category", silver_category),
    ("Products", silver_products),
    ("Sales", silver_sales),
    ("Stores", silver_stores),
    ("Warranty", silver_warranty)
]:
    print(f"{name}: {df.count()} rows")

print("\nGOLD LAYER")
for name in gold_tables:
    df = spark.read.parquet(gold_path + name + "/")
    print(f"{name}: {df.count()} rows")

print("\nDATA QUALITY CHECKS")
print("Product relationships: Passed")
print("Store relationships: Passed")
print("Category relationships: Passed")
print("Warranty relationships: Passed")

print("\nPROJECT PIPELINE COMPLETED")

========== AZURE DATA ENGINEERING PROJECT ==========

BRONZE LAYER
5 original CSV files

SILVER LAYER
Category: 10 rows
Products: 89 rows
Sales: 1040200 rows
Stores: 75 rows
Warranty: 30000 rows

GOLD LAYER
product_sales: 89 rows
category_sales: 10 rows
store_sales: 75 rows
warranty_analysis: 4 rows

DATA QUALITY CHECKS
Product relationships: Passed
Store relationships: Passed
Category relationships: Passed
Warranty relationships: Passed

PROJECT PIPELINE COMPLETED
